In [34]:
import pandas as pd
print(pd.__version__)

3.0.5


In [35]:
import os 
print(os.getcwd())

e:\Coading\job-posting-authenticity-detector\notebooks


In [36]:
emscad = pd.read_csv("../data/emscad_core.csv")
synth = pd.read_csv("../data/synthetic_stress_test.csv")

In [37]:
print("EMSCAD columns:", emscad.columns.tolist())
print("\nSynthetic columns:", synth.columns.tolist())

EMSCAD columns: ['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']

Synthetic columns: ['job_id', 'job_title', 'job_description', 'requirements', 'benefits', 'company_name', 'company_profile', 'industry', 'employment_type', 'location', 'salary_range', 'required_experience_years', 'education_level', 'department', 'posting_date', 'application_deadline', 'contact_email', 'company_website', 'has_logo', 'num_open_positions', 'job_function', 'telecommuting', 'fraud_reason', 'text_length', 'is_fake']


In [38]:
rename_map = {
    "job_title": "title",
    "job_description" : "description",
    "has_logo" : "has_company_logo",
    "job_function": "function",
    "is_fake" : "fraudulent",
    "education_level" : "required_education"
}

In [39]:
synth_renamed = synth.rename(columns=rename_map)

In [40]:
emscad["source"] =  "emscad"
synth_renamed["source"] = "synthetic"

In [41]:
# This shows us exact categorical lables EMSCAD uses
print(emscad["required_experience"].value_counts(dropna=False))

required_experience
NaN                 7050
Mid-Senior level    3809
Entry level         2697
Associate           2297
Not Applicable      1116
Director             389
Internship           381
Executive            141
Name: count, dtype: int64


In [42]:
print(synth["required_experience_years"].describe())

count    3000.000000
mean        5.044000
std         3.168292
min         0.000000
25%         2.000000
50%         5.000000
75%         8.000000
max        10.000000
Name: required_experience_years, dtype: float64


## The Bucketing of data (years of experience-> job roles)

since the emscad required ewperience goes to
required_experience
NaN                 7050
Mid-Senior level    3809
Entry level         2697
Associate           2297
Not Applicable      1116
Director             389
Internship           381
Executive            141

but require experience years in synthetic data only goes to max 10 years , that means directorial roles are not achievable , so we bucket roles to experince insted of dividing the roles to fit into experience brackets . this will be more appropriate for the data analysis but some rows of synthetic wil remain unmapped.

Proposed boundaries
Years	EMSCAD label
0–1	Entry level
2–4	Associate
5–10	Mid-Senior level

In [43]:
#Bucket Function
def bucket_experience(years):
    if years <=1:
        return "Entry level"
    elif years <=4:
        return "Associate"
    else :
        return "Mid-Senior level"
    
synth_renamed["required_experience"] = synth_renamed["required_experience_years"].apply(bucket_experience)
# print(synth_renamed["required_experience"].value_counts())
print(synth_renamed.columns.tolist())

['job_id', 'title', 'description', 'requirements', 'benefits', 'company_name', 'company_profile', 'industry', 'employment_type', 'location', 'salary_range', 'required_experience_years', 'required_education', 'department', 'posting_date', 'application_deadline', 'contact_email', 'company_website', 'has_company_logo', 'num_open_positions', 'function', 'telecommuting', 'fraud_reason', 'text_length', 'fraudulent', 'source', 'required_experience']


In [44]:
#Empty set confirms it — every label in your bucketed synthetic column now exists exactly in EMSCAD's vocabulary
print(set(synth_renamed["required_experience"].unique())-set(emscad["required_experience"].unique()))

set()


In [45]:
emscad["source"] =  "emscad"
synth_renamed["source"] =  "synthetic"

In [46]:
print(emscad.info())
print(synth_renamed.info())

<class 'pandas.DataFrame'>
RangeIndex: 17880 entries, 0 to 17879
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   job_id               17880 non-null  int64
 1   title                17880 non-null  str  
 2   location             17534 non-null  str  
 3   department           6333 non-null   str  
 4   salary_range         2868 non-null   str  
 5   company_profile      14572 non-null  str  
 6   description          17879 non-null  str  
 7   requirements         15184 non-null  str  
 8   benefits             10668 non-null  str  
 9   telecommuting        17880 non-null  int64
 10  has_company_logo     17880 non-null  int64
 11  has_questions        17880 non-null  int64
 12  employment_type      14409 non-null  str  
 13  required_experience  10830 non-null  str  
 14  required_education   9775 non-null   str  
 15  industry             12977 non-null  str  
 16  function             11425 non-nu

In [47]:
print(synth_renamed.columns.tolist())

['job_id', 'title', 'description', 'requirements', 'benefits', 'company_name', 'company_profile', 'industry', 'employment_type', 'location', 'salary_range', 'required_experience_years', 'required_education', 'department', 'posting_date', 'application_deadline', 'contact_email', 'company_website', 'has_company_logo', 'num_open_positions', 'function', 'telecommuting', 'fraud_reason', 'text_length', 'fraudulent', 'source', 'required_experience']


In [48]:
synth_renamed = synth.rename(columns=rename_map)
synth_renamed["source"] = "synthetic"
print(synth_renamed.columns.tolist())

['job_id', 'title', 'description', 'requirements', 'benefits', 'company_name', 'company_profile', 'industry', 'employment_type', 'location', 'salary_range', 'required_experience_years', 'required_education', 'department', 'posting_date', 'application_deadline', 'contact_email', 'company_website', 'has_company_logo', 'num_open_positions', 'function', 'telecommuting', 'fraud_reason', 'text_length', 'fraudulent', 'source']


# messiness of dataset
look at how differently missing values are distributed between the two datasets. EMSCAD has real gaps everywhere — salary_range only 2,868/17,880 filled, benefits 10,668/17,880, department 6,333/17,880. The synthetic dataset has almost no missing values except company_name, company_website, and fraud_reason. That's not a coincidence — it's a direct fingerprint of the two datasets' origins: EMSCAD is messy because real recruiters fill out forms inconsistently; the synthetic data is clean because whatever generated it didn't model that real-world sloppiness. That's actually a good observed insight to write down now — it strengthens your earlier point about treating synthetic data as a separate stress-test set rather than blending it in as if it were equally realistic.

In [49]:
emscad["salary_missing"] = emscad["salary_range"].isnull()
print(emscad.groupby("salary_missing")["fraudulent"].mean())

salary_missing
False    0.077755
True     0.042832
Name: fraudulent, dtype: float64


# Strange finding
Read it carefully: salary_missing = False means salary is present, and that group has the higher fraud rate (7.78%) compared to salary_missing = True (missing salary, only 4.28%). That's the opposite of what "missing salary = vague/suspicious" would predict.

Think about why, using the same reasoning we used for has_company_logo: scammers want to lure you in with something concrete and appealing — an unrealistically high salary is classic bait ("$5000/month, work from home, no experience needed"). Legitimate companies, especially ones being careful and professional, often deliberately withhold salary until later stages, exactly like your hypothesis said — but that same caution is actually a sign of legitimacy, not vagueness. Meanwhile, fraudulent postings tend to flaunt salary because that's the whole hook.

In [50]:
# This prints 10 examples of salary that are not null.
print(emscad[emscad["salary_missing"]== False]["salary_range"].head(10))

6       20000-28000
10    100000-120000
15    120000-150000
23    100000-120000
31      50000-65000
42      40000-50000
65            60-80
76      65000-70000
77           75-115
79     75000-110000
Name: salary_range, dtype: str


# observation on salary
the smaller ranges like 60 -80 , 75-115 are not broken but hourly rates.

In [55]:
bad_rows = salaries[salaries.str.contains("[A-Za-z]", na=False)]
print(bad_rows.head(20))
print(f"\nTotal bad rows: {len(bad_rows)}")

159       9-Dec
1884      3-Apr
1981      4-Apr
2313     Oct-15
4299      8-Sep
9124      4-Jun
9902     10-Oct
9911     Oct-20
10316    Jun-18
10785    10-Oct
10788    11-Nov
10860    10-Nov
10883    10-Oct
10889    10-Nov
10896    10-Oct
10905    10-Nov
11361    11-Dec
11495     2-Apr
11606    10-Nov
12421    10-Oct
Name: salary_range, dtype: str

Total bad rows: 26


In [ ]:
# fix for Excel date-corruption pattern
clean_salaries = salaries[~salaries.str.contains("[A-Za-z]", na=False)]
lower_bound = clean_salaries.str.split("-").str[0].astype(float)
print(lower_bound.describe())

count    2.842000e+03
mean     5.154100e+05
std      1.769313e+07
min      0.000000e+00
25%      1.800000e+04
50%      3.500000e+04
75%      6.000000e+04
max      8.000000e+08
Name: salary_range, dtype: float64


In [57]:
print(lower_bound.sort_values().head(80).tail(50))

12058    0.0
5734     0.0
810      0.0
8910     0.0
12417    0.0
12419    0.0
12424    0.0
3475     0.0
11451    0.0
11379    0.0
3564     0.0
707      0.0
4293     0.0
3561     0.0
3642     0.0
12284    0.0
12088    0.0
12119    0.0
12434    0.0
7354     0.0
763      0.0
767      0.0
12501    0.0
12790    0.0
12849    0.0
5821     0.0
11255    0.0
4306     0.0
11229    0.0
11203    0.0
12389    0.0
16851    0.0
3442     0.0
12596    0.0
7837     0.0
13523    0.0
2917     0.0
4375     0.0
10997    0.0
11068    0.0
10885    0.0
10887    0.0
10902    0.0
10911    0.0
10918    0.0
5797     0.0
12857    0.0
12704    0.0
12947    0.0
5904     0.0
Name: salary_range, dtype: float64


In [58]:
zero_count = (lower_bound == 0).sum()
print(f"Total zero values: {zero_count}")

print(lower_bound[lower_bound > 0].sort_values().head(30))

Total zero values: 195
14769     7.0
17538    13.0
3345     13.0
4043     13.0
17722    13.0
8162     13.0
1662     13.0
17195    13.0
8476     14.0
17698    15.0
6228     15.0
17700    15.0
16765    15.0
17783    15.0
17389    16.0
9332     17.0
7818     17.0
17775    17.0
17755    17.0
3342     18.0
7160     20.0
17638    20.0
14063    20.0
17666    20.0
3786     20.0
2930     20.0
6122     20.0
1843     21.0
4565     21.0
17724    21.0
Name: salary_range, dtype: float64


In [62]:
lower_bound_clean = lower_bound[lower_bound >= 1000]
print(lower_bound_clean.describe())
print(f"\nRows excluded as likely non-annual: {(lower_bound < 1000).sum()}")

count    2.446000e+03
mean     5.988461e+05
std      1.907089e+07
min      1.000000e+03
25%      2.500000e+04
50%      4.000000e+04
75%      6.500000e+04
max      8.000000e+08
Name: salary_range, dtype: float64

Rows excluded as likely non-annual: 396


**Salary data cleaning decision:** Excluded 26 rows with Excel date-corrupted values 
(e.g. "Oct-15" instead of "10-15"). Excluded 396 additional rows with lower-bound 
salary under 1000 as likely non-annual pay units (hourly/weekly); exact unit could 
not be reliably determined from the data alone. Remaining outliers (e.g. max of 
800M) still need investigation.